# Arquitectura

<img src="capturas/infografia-solucion-dav.png" alt="arquitectura" width="1000" height="500">

In [ ]:
import os
import pandas as pd

# 1. Integraciones de OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 2. Base de Datos Vectorial -> USAMOS FAISS (Plan B)
from langchain_community.vectorstores import FAISS

# 3. Componentes del Núcleo
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- CONFIGURACIÓN ---
# Genrar el Toket OpennIA
os.environ["OPENAI_API_KEY"] = "tu_api_key_aqui"

# Modelo LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


In [2]:

class MockDatabase:
    """Simula la base de datos transaccional del Banco/Ecommerce"""
    def __init__(self):
        # 1. USUARIOS (Segmentación y Financiero)
        self.usuarios = [
            {"id": "user123", "nombre": "Juan Pérez", "segmento": "VIP", "saldo_tarjeta": 5000000},
            {"id": "user456", "nombre": "Maria Lopez", "segmento": "Estándar", "saldo_tarjeta": 1500000}
        ]
        
        # 2. PRODUCTOS (Catálogo enriquecido para IA)
        self.productos = [
            {"id": 1, "nombre": "Laptop Gaming ASUS ROG", "categoria": "Laptops", "precio": 4500000, "stock": 5, "desc": "RTX 3060, 16GB RAM. Ideal gamers.", "sugeridos": [3, 4]},
            {"id": 2, "nombre": "MacBook Air M2", "categoria": "Laptops", "precio": 5200000, "stock": 2, "desc": "Chip M2, Ligero, 13.6 pulgadas.", "sugeridos": [4]},
            {"id": 3, "nombre": "Mouse Logitech MX Master 3", "categoria": "Periféricos", "precio": 450000, "stock": 50, "desc": "Ergonómico, Bluetooth.", "sugeridos": []},
            {"id": 4, "nombre": "Monitor Samsung Odyssey G5", "categoria": "Monitores", "precio": 1800000, "stock": 10, "desc": "27 pulgadas, 144Hz curvo.", "sugeridos": []}
        ]
        
        # 3. PEDIDOS (Historial para Post-Venta)
        self.pedidos = [
            {"id": "PED001", "usuario_id": "user123", "items": [1], "total": 4500000, "estado": "Entregado", "fecha": "2024-01-15"},
            {"id": "PED002", "usuario_id": "user123", "items": [3], "total": 450000, "estado": "En tránsito", "fecha": "2024-02-01"}
        ]
        
        # Crear DataFrame para búsqueda vectorial de productos
        self.df_productos = pd.DataFrame(self.productos)
        self.df_productos['texto_busqueda'] = self.df_productos.apply(
            lambda x: f"ID:{x['id']} | Producto: {x['nombre']} | Precio: ${x['precio']} | Info: {x['desc']}", axis=1
        )

# Inicializamos la DB
db = MockDatabase()
print("Base de datos relacional simulada cargada.")

Base de datos relacional simulada cargada.


In [3]:
class DaviviendaAgent:
    def __init__(self, database):
        self.db = database
        self.vectorstore = None
        self.retriever = None
        self._build_vector_store()
        
    def _build_vector_store(self):
        """Crea el índice semántico de productos usando FAISS (Sin errores de SQLite)"""
        print("⏳ Indexando catálogo de productos...")
        self.vectorstore = FAISS.from_texts(
            texts=self.db.df_productos['texto_busqueda'].tolist(),
            embedding=OpenAIEmbeddings()
        )
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
        print("Indexación completada.")

    def _get_user_context(self, user_id):
        """Recupera la 'ficha del cliente' en tiempo real"""
        # Buscar usuario
        user = next((u for u in self.db.usuarios if u['id'] == user_id), None)
        if not user: return "Usuario no identificado."
        
        # Buscar sus pedidos
        pedidos = [p for p in self.db.pedidos if p['usuario_id'] == user_id]
        
        # Formatear resumen para el LLM
        resumen = f"""
        - Cliente: {user['nombre']} ({user['segmento']})
        - Saldo Disponible: ${user['saldo_tarjeta']}
        - Historial de Pedidos Recientes: {pedidos}
        """
        return resumen

    def responder(self, user_id, pregunta):
        """Método principal: Orquesta la recuperación de datos y la generación de respuesta"""
        
        # 1. Recuperar contexto del usuario (Datos Estructurados)
        contexto_usuario = self._get_user_context(user_id)
        
        # 2. Recuperar productos relevantes (Datos No Estructurados - RAG)
        docs_relevantes = self.retriever.invoke(pregunta)
        contexto_productos = "\n".join([d.page_content for d in docs_relevantes])
        
        # 3. Construir el Prompt Híbrido
        template = """Eres un Asesor Experto de Davivienda. Tu objetivo es dar soluciones integrales.
        
        DATOS DEL CLIENTE (Úsalos para verificar saldo o estado de pedidos):
        {user_context}
        
        CATÁLOGO DE PRODUCTOS RELEVANTES (Úsalos para recomendar):
        {product_context}
        
        PREGUNTA DEL CLIENTE:
        {question}
        
        INSTRUCCIONES:
        - Si preguntan por pedidos, usa los DATOS DEL CLIENTE.
        - Si piden comprar, verifica si su 'Saldo Disponible' es suficiente contra el precio del producto.
        - Si recomiendas, sugiere complementos lógicos.
        - Responde de forma ejecutiva y amable.
        """
        
        prompt = ChatPromptTemplate.from_template(template)
        
        # 4. Invocar al LLM
        chain = prompt | llm | StrOutputParser()
        
        respuesta = chain.invoke({
            "user_context": contexto_usuario,
            "product_context": contexto_productos,
            "question": pregunta
        })
        
        return respuesta

# Instanciamos el Agente
agente = DaviviendaAgent(db)

⏳ Indexando catálogo de productos...
Indexación completada.


In [4]:
# Simulamos que somos el usuario "Juan Pérez" (user123)
USER_ID = "user123"

print("--- ESCENARIO 1: Post-Venta y Rastreo (Uso de Historial) ---")
preg1 = "¿Qué ha pasado con el mouse que compré?"
print(f" Usuario: {preg1}")
print(f" Agente: {agente.responder(USER_ID, preg1)}")
print("\n" + "="*50 + "\n")

print("--- ESCENARIO 2: Validación Financiera (Cruce de Saldo vs Precio) ---")
preg2 = "Quiero comprar el MacBook Air, ¿me alcanza el cupo de mi tarjeta?"
print(f" Usuario: {preg2}")
print(f" Agente: {agente.responder(USER_ID, preg2)}")
print("\n" + "="*50 + "\n")

print("--- ESCENARIO 3: Recomendación Cruzada (Contexto de Producto) ---")
preg3 = "Busco un monitor bueno para trabajar, ¿qué me recomiendas?"
print(f" Usuario: {preg3}")
print(f" Agente: {agente.responder(USER_ID, preg3)}")

--- ESCENARIO 1: Post-Venta y Rastreo (Uso de Historial) ---
 Usuario: ¿Qué ha pasado con el mouse que compré?
 Agente: ¡Hola Juan Pérez! 

Gracias por tu pregunta. Según tu historial de pedidos, el mouse que compraste (ID:3) está actualmente "En tránsito" y se espera que llegue pronto. Si deseas más información sobre el estado de entrega, por favor házmelo saber y con gusto te ayudaré a rastrearlo.

Además, si estás interesado en complementar tu compra con algún otro producto, te recomendaría considerar el MacBook Air M2 (ID:2) que es ligero y potente, ideal para complementar tu setup. Tu saldo disponible es de $5000000, por lo que puedes adquirirlo sin problema.

Quedo atento a cualquier otra consulta que tengas. ¡Que tengas un excelente día!


--- ESCENARIO 2: Validación Financiera (Cruce de Saldo vs Precio) ---
 Usuario: Quiero comprar el MacBook Air, ¿me alcanza el cupo de mi tarjeta?
 Agente: ¡Hola Juan Pérez! Como Asesor Experto de Davivienda, estoy aquí para ayudarte. 

Para re